In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.io import  wavfile
from IPython import display
import json, os

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Compute normalized cross-correlation
def normalized_cross_correlation(x1, x2):
    n1 = len(x1)
    result = []
    for i in range(len(x2) - n1 + 1):
        segment = x2[i:i + n1]
        numerator = np.sum((x1 - np.mean(x1)) * (segment - np.mean(segment)))
        denominator = np.sqrt(np.sum((x1 - np.mean(x1))**2) * np.sum((segment - np.mean(segment))**2))
        result.append(numerator / denominator if denominator != 0 else 0)
    return np.array(result)

def find_sub_array(target, long_array):
  correlation = normalized_cross_correlation(target, long_array)
  
  # Find the starting index where target best matches long_array
  best_start_index = np.argmax(correlation)
  
  # Plot normalized cross-correlation
  # plt.figure(figsize=(12, 5))
  plt.subplot(2, 1, 1)
  plt.plot(correlation, label='Normalized Cross-correlation')
  plt.axvline(best_start_index, color='r', linestyle='--', label=f'Best match: {best_start_index}')
  plt.title('Normalized Cross-correlation between target and long_array')
  plt.legend()
  
  # Plot long_array and overlay target at the best match position
  plt.subplot(2, 1, 2)
  plt.plot(long_array, label='long_array', marker='.', color='gray', alpha=0.7)
  plt.plot(
      range(best_start_index, best_start_index + len(target)),
      target,
      label='target (best match)',
      marker='o',
      color='red'
  )
  plt.title('long_array with target overlayed at best match position')
  plt.legend()
  plt.tight_layout()
  rms_error = np.sqrt(np.mean((long_array[best_start_index:best_start_index + len(target)] - target)**2))
  print(f"RMS Error of sub-string = {rms_error}")


# Example data
plt.figure(figsize=(6,4))
x_long = np.random.rand(100)  # Longer array
x_short = x_long[62:75] # + np.random.normal(0, 0.05, len(x_short))  # Shorter array, with some noise
find_sub_array(x_short, x_long)


In [ ]:
# wavfile.write?
# wavfile.read?
fs, wav_from_file = wavfile.read("/Users/jeremy/tmp/happy_2ch.wav")
print(f"fs={fs}, shape = {wav_from_file.shape}")

In [ ]:
pwd

In [ ]:
wav_dir = os.getenv("HOME")
wav_dir = "."
with open(os.path.join(wav_dir, "wav.json"), "r") as fpi:
  recorded_wav_stereo = json.load(fpi)
  recorded_wav_stereo = np.array(recorded_wav_stereo)
  

In [ ]:
print(recorded_wav_stereo[:12])
print(wav_from_file[:12,0])


In [ ]:
num_channels = 2
num_samples = len(recorded_wav_stereo)//num_channels
recorded_wav = np.reshape(recorded_wav_stereo[0:num_channels*num_samples], (num_samples, num_channels))
print(f"recorded wav shape = {recorded_wav.shape}")

In [ ]:
num_samples = recorded_wav.shape[0]
print(f"Captured {num_samples} samples")

In [ ]:

plt.subplot(3,1,1)
plt.plot(wav_from_file[:num_samples,0])
plt.subplot(3,1,2)
plt.plot(recorded_wav[:num_samples,0])
plt.subplot(3,1,3)
plt.plot(wav_from_file[:num_samples,0] - recorded_wav[:num_samples,0])

In [ ]:
find_sub_array(recorded_wav[0:128,0], wav_from_file[:,0])

In [ ]:
find_sub_array(recorded_wav[129:256,0], wav_from_file[:,0])

In [ ]:
def generate_sine_wave(frequency=200, sample_rate=16000, sig_len=2048, amplitude=1.0):
    """
    Generate a sine wave and write to a C array.
    
    Args:
        frequency (float): Frequency of the sine wave in Hz.
        sample_rate (int): Sampling rate in samples per second (Hz).
        duration (float): Duration of the sine wave in seconds.
        amplitude (int): Maximum amplitude of the sine wave (for 16-bit signed values).

    Returns:
        None
    """
    # Generate time values
    t = np.arange(0, int(sig_len))/sample_rate
    
    # Generate sine wave
    sine_wave = (amplitude * np.sin(2 * np.pi * frequency * t)).astype(np.float32)
    # Create the C array string
    c_array_str = f"const float32_t sine_wave[{len(sine_wave)}] = {{\n"
    c_array_str += ",\n".join(
        ", ".join(f"{x:5.4}" for x in sine_wave[i:i+8]) 
        for i in range(0, len(sine_wave), 8)
    )
    c_array_str += "\n};\n"
    
    # Write the C array to a file
    # with open("sine_wave_array.h", "w") as f:
    print(c_array_str)

    # print("C array written to sine_wave_array.h")

# Generate and save the sine wave
generate_sine_wave(frequency=7992)

In [ ]:
def generate_cmplx_sine_wave(frequency=200, sample_rate=16000, sig_len=1024, amplitude=1.0):
    """
    Generate a sine wave and write to a C array.  Array will be complex but with all imaginary values
    set to 0.  Structured as [x_real_0, x_imag_0, x_real_1, x_imag_1, ....]
    
    Args:
        frequency (float): Frequency of the sine wave in Hz.
        sample_rate (int): Sampling rate in samples per second (Hz).
        sig_len (int): number of time samples.  Output array will be 2x this length
        amplitude (int): Maximum amplitude of the sine wave (for 16-bit signed values).

    Returns:
        None
    """
    # Generate time values
    t = np.arange(0, int(sig_len))/sample_rate
    
    # Generate sine wave
    sine_wave = (amplitude * np.sin(2 * np.pi * frequency * t)).astype(np.float32)
    sine_wav_cplx = np.zeros(2*sig_len)
    sine_wav_cplx[0:-1:2] = sine_wave
    # Create the C array string
    c_array_str = f"const float32_t sine_wave[{len(sine_wav_cplx)}] = {{\n"
    c_array_str += ",\n".join(
        ", ".join(f"{x:5.4}" for x in sine_wav_cplx[i:i+8]) 
        for i in range(0, len(sine_wav_cplx), 8)
    )
    c_array_str += "\n};\n"
    
    # Write the C array to a file
    # with open("sine_wave_array.h", "w") as f:
    print(c_array_str)

    # print("C array written to sine_wave_array.h")

# Generate and save the sine wave
generate_cmplx_sine_wave(frequency=7992)

In [ ]:
freq = np.linspace(0,8000, 512)
hAx = plt.gca()
hAx.plot(freq, fft_mag[0:512])
hAx.set_xlim(0,400)
plt.grid(True)